# **The Baltic Sentinel: Evaluating Estonia’s Energy Resilience**
**Course:** 42578 Advanced Business Analytics


## **1. Introduction - Ursula** 


In early 2026, Estonia's electricity system faced a severe stress event driven by extreme cold, weak wind generation, and major production outages.[1][2][3] This crisis pushed resilience margins to their limits, triggering record-high prices that peaked at 505.06 €/MWh in January and escalated to 655.48 €/MWh by February.[4]

January 2026 was exceptional because multiple stressors aligned perfectly. Decades-low temperatures across the Baltics and Finland coincided with record demand, weak wind, and outages, creating an ideal real-world case study.[3][4] At a strategic level, Estonia's energy transition is moving away from legacy fossil generation toward renewables and modernised infrastructure. The tension between preserving grid reliability and accelerating this transition makes the 2026 crisis especially relevant for evaluating security of supply.[7]

This project analyses that episode as a resilience problem rather than solely a pricing event. The key issue is not just why prices became so high, but what the event reveals about Estonia's ability to maintain a secure electricity supply under disruption. Estonia's system has historically depended on a combination of domestic generation, cross-border imports, and a growing share of renewables, meaning that its stability depends on both internal production and regional interconnection.

A central motivation for this project is the public debate around whether more wind capacity could have changed the outcome. In February 2026, Utilitas Wind published a modelled scenario arguing that the Liivi Bay offshore project could have halved January prices and saved society €75 million, framing wind capacity as a critical resilience measure rather than just a decarbonisation tool.[5]

**Our project therefore asks two linked questions:** 
* **1) How exposed was Estonia's power system during the January 2026 crisis if imports were unavailable?**
* **2) Could planned wind farm expansions have materially improved resilience during those critical hours?**

These questions matter because resilience is determined by system performance in the worst hours, not by average performance in normal conditions.

To answer this, the project combines historical analysis, forecasting, and counterfactual simulation. On the supply side, a spatio-temporal graph neural network is used to model electricity production under different grid states and wind scenarios. On the demand side, a separate forecasting framework estimates hourly consumption and its uncertainty under January 2026 conditions, after which supply and demand are compared under baseline, isolation, and wind-expansion scenarios.

This makes the project more than a forecasting exercise. By combining public reporting, industry claims, and scenario modelling, the project evaluates whether new wind investments would only lower emissions and average prices in normal times, or whether they could also strengthen security of supply during crisis conditions.[5][6]

### References

[1]: ERR News. *January electricity prices highest Estonia has seen in years*. 29 Jan 2026. [https://news.err.ee/1609927241/january-electricity-prices-highest-estonia-has-seen-in-years](https://news.err.ee/1609927241/january-electricity-prices-highest-estonia-has-seen-in-years)

[2]: ERR News. *Cold weather driving up price of electricity, putting production capacity to the test*. 2 Feb 2026. [https://news.err.ee/1609930625/cold-weather-driving-up-price-of-electricity-putting-production-capacity-to-the-test](https://news.err.ee/1609930625/cold-weather-driving-up-price-of-electricity-putting-production-capacity-to-the-test)

[3]: Elenger. *Power market overview Q1 2026*. 13 Apr 2026. [https://elenger.ee/en/power-market-overview-q1-2026/](https://elenger.ee/en/power-market-overview-q1-2026/)

[4]: Konkurentsiamet. *Energiaturgude ülevaade veebruar 2026*. April 2026. [https://www.konkurentsiamet.ee/sites/default/files/documents/2026-04/Energiaturgude%20%C3%BClevaade%20veebruar%202026.pdf](https://www.konkurentsiamet.ee/sites/default/files/documents/2026-04/Energiaturgude%20%C3%BClevaade%20veebruar%202026.pdf)

[5]: Ärileht / Delfi. *Utilitas Windi juht: tuulepargid võinuks jaanuari elektrihinna pooleks lüüa ja säästa kümneid miljoneid*. 17 Feb 2026. <https://arileht.delfi.ee/artikkel/120436765/utilitas-windi-juht-tuulepargid-voinuks-jaanuari-elektrihinna-pooleks-luua-ja-saasta-kumneid-miljoneid>

[6]: European Commission. *Estonia's recovery and resilience plan*. [https://reforms-investments.ec.europa.eu/recovery-and-resilience-facility-1/country-pages/estonias-recovery-and-resilience-plan_en](https://reforms-investments.ec.europa.eu/recovery-and-resilience-facility-1/country-pages/estonias-recovery-and-resilience-plan_en)

In [ ]:
# Standard Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Note: Custom helper functions and modular scripts (STGNN, PowerScaler, etc.) 
# are hosted in the sandra359/advanced_ba repository.
import sys
import os

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## **2. Data - Sofie**

Our analysis uses a modular pipeline to isolate and recombine the two sides of the grid equation. 

### **2.1 Data Sources**
*   **Grid Operations (Elering):** Hourly production, consumption, and cross-border flow data[cite: 3].
*   **Market Pricing (Nord Pool):** Day-Ahead clearing prices for the EE, FI, and LV bidding zones[cite: 3].
*   **Environment (OpenWeather API):** Historical temperature and wind speeds for specific coordinates[cite: 3].

### **2.2 Wind Capacity Scenarios** 
*   **Baseline (Current):** **694 MW** of grid-connected wind[cite: 1].
*   **Scenario A (Established):** Adding **323 MW** from Pärnu and Aidu[cite: 1].
*   **Scenario B (Full Pipeline):** Adding a total of **887 MW** of new capacity[cite: 1].

In [ ]:
# Define capacities based on Project Advanced BA
CURRENT_WIND_MW = 694.0
SCENARIO_A_MW = CURRENT_WIND_MW + 323.0
SCENARIO_B_MW = CURRENT_WIND_MW + 887.0

# Physical Grid Limits (MW)
ESTLINK_CAPACITY = 1016  
LATVIA_CAPACITY = 700    

print(f"Baseline Wind: {CURRENT_WIND_MW} MW")
print(f"Scenario A (Established) Wind: {SCENARIO_A_MW} MW")
print(f"Scenario B (Full Pipeline) Wind: {SCENARIO_B_MW} MW")

## **3. Exploratory Data Analysis: Visualizing the Crisis**
Before modeling, we must understand the magnitude of the "Perfect Storm" in January 2026. Here, we visualize the temperature-demand trap, the EstLink failure cliff, and regional price divergence.

In [ ]:
# Load historical data (placeholder paths - update with actual data from sandra359/advanced_ba)
# df_historical = pd.read_csv('data/historical_jan2026.csv')

# TODO: Plot 1: Temperature vs. Consumption Spikes
# TODO: Plot 2: EstLink Flow (showing the drop to 0 MW)
# TODO: Plot 3: Nord Pool Price Divergence (EE vs FI vs LV)

### **TODO:**

* how much import we have right now (plots) - Sandra & Tomas
* Estonian electricity generation by type - Ursula 
* Prices over time - Tomas

## **4. Demand Analysis: Probabilistic Forecasting - Sofie**

Resilience is tested against extremes. We use an **ARIMAX** model to predict hourly consumption, accounting for temperature and temporal features[cite: 3].

To identify "edge-case" demand, we perform a **Monte Carlo simulation** using the model’s residual standard deviation. This allows us to extract a **P95 Worst-Case Demand** curve, ensuring our stress test targets a 1-in-100-year consumption spike[cite: 3].

In [ ]:
# The ARIMAX model and Monte Carlo simulation logic are executed in Demand.py.
# Here, we load the pre-calculated P95 Worst-Case Demand curve for January 2026.

# demand_df = pd.read_csv("data/demand_p95_jan2026.csv")
# demand_df['timestamp'] = pd.to_datetime(demand_df['timestamp'])

# TODO: Visualize the P50 (Average) vs P95 (Worst-Case) Demand Forecast

### TODO

* add all the text from Sofie

## **5. Wind Counterfactual Simulation - Ursula**

We translate historical wind speeds into energy production using standardized turbine power curves (e.g., Vestas V150)[cite: 3]. We adjust for air density based on temperature.

These hourly production profiles act as the inputs for Scenario A and Scenario B.

In [ ]:
# Wind scenario generation logic is hosted in ursula_wind_counterfactual.ipynb
# Load the pre-calculated wind production scenarios

# df_wind = pd.read_csv("data/wind_production_scenarios.csv")

# TODO: Plot the simulated wind generation for Baseline, Scenario A, and Scenario B

## **6. Supply Analysis: Spatio-Temporal GNN - Sandra & Tomas**

The Estonian grid is a geographic network. We use a **Spatio-Temporal Graph Neural Network (ST-GNN)** to model how energy flows and market prices propagate across nodes[cite: 3].

*   **Quantile Regression:** The GNN natively outputs **P10 (Worst-Case Supply)**, P50, and P90, handling supply-side uncertainty without a second Monte Carlo simulation[cite: 3]. 
*   **"Market Blindness":** To simulate the crisis, we sever the graph edges to Finland. This forces the model to predict domestic production in **Island Mode**, reacting only to local weather rather than Nordic market prices[cite: 3].

In [ ]:
# ST-GNN training and inference are executed in Supply_wind.py.
# We load the output predictions for the 4 scenarios.

try:
    supply_df = pd.read_csv("data/gnn_supply_scenarios_jan2026.csv")
    supply_df['timestamp'] = pd.to_datetime(supply_df['timestamp'])
    print("GNN Supply Scenarios Loaded Successfully.")
except FileNotFoundError:
    print("Placeholder: Awaiting output from Supply_wind.py")

# TODO: Visualize the P10 Supply Baseline vs P10 Supply with +887MW Wind

## **7. Resilience Simulation: The Balance Equation - Sandra & Tomas**

This is the final "Stress Test" where we combine our independent models to calculate the physical blackout threshold.

We calculate the hourly deficit under isolation using the formula:
**Hourly Deficit = Demand(P95) - Domestic Production(P10)**

We apply the physical constraints of the grid:
*   **Scenario 1:** Full connectivity (EstLink + Latvia).
*   **Scenario 2:** **EstLink Cut** (Only Latvia capacity available).
*   **Scenarios 3 & 4:** **EstLink Cut** + New Wind Production.

The primary metric is **Total Energy Unserved (GWh)**—the amount of demand that would have resulted in physical blackouts[cite: 3].

In [ ]:
# --- THE BALANCE EQUATION ---
# Assuming supply_df and demand_df are loaded and merged

'''
results = []
for index, row in supply_df.iterrows():
    demand = row['demand_p95']
    
    # Calculate raw deficits (Demand - Domestic Supply)
    def_s1 = max(0, demand - row['supply_s1_baseline'])
    def_s2 = max(0, demand - row['supply_s2_isolated'])
    def_s3 = max(0, demand - row['supply_s3_wind323'])
    def_s4 = max(0, demand - row['supply_s4_wind887'])
    
    # Apply Physical Import Limits
    blackout_s1 = max(0, def_s1 - (ESTLINK_CAPACITY + LATVIA_CAPACITY))
    
    # Isolated crisis (EstLink is CUT, only Latvia remains)
    blackout_s2 = max(0, def_s2 - LATVIA_CAPACITY)
    blackout_s3 = max(0, def_s3 - LATVIA_CAPACITY)
    blackout_s4 = max(0, def_s4 - LATVIA_CAPACITY)
    
    results.append({
        'timestamp': row['timestamp'],
        'demand': demand,
        'blackout_s1': blackout_s1,
        'blackout_s2': blackout_s2,
        'blackout_s3': blackout_s3,
        'blackout_s4': blackout_s4
    })

sim_df = pd.DataFrame(results)
'''

# TODO: Print Total Energy Unserved (MWh) for each scenario
# TODO: Plot Hourly Power Shortages (S2 vs S4)

## **8. Conclusion & Recommendations - Ursula**

Our findings demonstrate that while Estonia achieved technical disconnection from BRELL in 2025, true resilience against "Price Stress" and supply shocks depends heavily on the speed of Scenario B’s implementation[cite: 1, 3]. 

*   **Key Finding:** The addition of **887 MW** of wind capacity reduces the risk of physical blackouts during a January 2026-style "Island Mode" event by **[X]%**.
*   **Strategic Autonomy:** By building the capacity to operate autonomously during severe grid fragmentations, Estonia ensures it can withstand, adapt to, and recover from multi-front infrastructure crises[cite: 2].